# 2. FranceCrops: prepare references and collect annual imagery

Use declared cabbage and table-potato parcels. This source has no carrots; original interpolated imagery arrays are not needed.


## 1. Setup and configuration

Choose a Python 3.12+ kernel, review the settings below, then **Restart Kernel → Run All**. Opening the notebook starts no work. Existing settings, selections, archives and jobs are preserved.

- `reuse`: use local references; satellite collection is enabled separately.
- `download_missing`: obtain missing references within the source limits.
- `offline`: use local references, with no package installation, sign-in or satellite calls. Dependencies must already be installed.

Use `download_missing` on a fresh clone. Enter an Earth Engine-enabled project ID when prompted. All sources target **up to 100 candidate parcels per available crop per round**; existing larger batches remain intact. Saved outputs may describe earlier runs until cells are rerun.


In [1]:
# Review these settings before Run All.
reference_mode = "reuse"  # reuse | download_missing | offline
run_satellite_collection = True  # offline always disables satellite calls

from pathlib import Path
import os
import sys
repository_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), Path.cwd())
source_folder = str(repository_root / "src")
if source_folder not in sys.path:
    sys.path.insert(0, source_folder)
from crop_pipeline import notebook_setup, source_workflow
from crop_pipeline.notebook_display import show_table, show_mapping, show_items
reference_options = source_workflow.reference_options(reference_mode)
notebook_setup.prepare_environment(repository_root, install_missing=reference_mode != "offline")
os.chdir(repository_root)

from crop_pipeline.sources import adaptive, annual
from crop_pipeline import progress as pipeline_progress
from crop_pipeline.windows.seasons import file_sha256
settings_path = Path("settings.local.json")
local_settings = notebook_setup.ensure_local_settings(settings_path)
report_folder = Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).expanduser().resolve()
annual_archive_root = annual.archive_root(local_settings, report_folder)
run_name = adaptive.COLLECTION_NAME
candidate_parcels_per_crop = source_workflow.ROUND_PARCELS_PER_CROP
run_satellite_collection = run_satellite_collection and reference_mode != "offline"
authenticate_now = run_satellite_collection
download_workers = 2
intervals_per_request = 6

from crop_pipeline.sources import francecrops as francecrops_workflow
source_name = "francecrops"
inward_buffer_metres = 10  # Polygon interiors, preserving the existing rule.
revision = francecrops_workflow.DEFAULT_REVISION
source_table_folder = report_folder.parent / "francecrops/source_metadata" / revision

# Resolve account configuration before source transfers; authentication occurs during collection.
earth_engine_project = notebook_setup.project_for_collection(local_settings, settings_path, required=run_satellite_collection)
print(f"Source: {source_name} | References: {reference_mode} | Round limit: {candidate_parcels_per_crop} per crop")
print(f"Inward buffer: {inward_buffer_metres} m | Satellite collection: {run_satellite_collection}")
print("Archive:", annual_archive_root)
print("Backend: Earth Engine batch assets → Parquet transport → annual TIFFs")
notebook_progress = pipeline_progress.start_notebook(
    '02_francecrops_preparation', source_workflow.STAGES, report_folder.parent / "progress",
    workload={"folder": str(annual_archive_root), "source": source_name},
)


Notebook environment ready: /Users/josedavid/Documents/programming/external-crop-data-classification/.venv/bin/python
Collection: adaptive_training
Archive root: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_archive
Satellite collection enabled: True


/Users/josedavid/Documents/programming/external-crop-data-classification/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
02_francecrops_preparation: stages:   0%|          | 0/4 [00:01<?, ?stage/s, elapsed 0m 01s; notebook ETA learning (future stages not yet timed)]

02_francecrops_preparation: stages:  75%|███████▌  | 3/4 [00:00<00:00,  4.66stage/s, elapsed 0m 02s; notebook ETA learning (future stages not yet timed)]      

Trusted certificates: 121


## 2. Obtain references

Reuse the pinned class-map, label and footprint Parquet tables. `download_missing` retrieves missing tables only. Original interpolated imagery arrays are not downloaded.


In [2]:
pipeline_progress.begin_stage("Obtain references")
table_paths = francecrops_workflow.download_reference_tables(
    source_table_folder, revision=revision,
    download_if_missing=reference_options["download_if_missing"],
)
reference_note = f"Pinned FranceCrops release {revision}; cabbage/table potato only"
show_mapping(table_paths, title="Reference files", key_label="Reference", value_label="File")
pipeline_progress.end_stage()


02_francecrops_preparation: stages:  25%|██▌       | 1/4 [00:00<00:00, 3095.43stage/s, elapsed 0m 01s; notebook ETA learning (future stages not yet timed)]        

Verified reference paths: {'class_map': PosixPath('/Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/francecrops/source_metadata/6b6c51b780d8579138593f92738ed52f2528096e/data/protocol/class_map.parquet'), 'metadata': PosixPath('/Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/francecrops/source_metadata/6b6c51b780d8579138593f92738ed52f2528096e/data/protocol/metadata.parquet'), 'geolocation': PosixPath('/Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/francecrops/source_metadata/6b6c51b780d8579138593f92738ed52f2528096e/data/geolocation/geolocation.parquet')}


## 3. Validate references

Join explicit crop labels to parcel boundaries. Preserve source-split, identity, geometry and label checks; only eligible training references enter candidate selection. Use the existing **10 m inward buffer** on parcel polygons.


In [ ]:
pipeline_progress.begin_stage("Validate references")
reference_features, reference_audit = francecrops_workflow.load_target_parcels(table_paths, revision, return_audit=True)
show_mapping(reference_audit, title="Reference audit", key_label="Check", value_label="Result")
print("Edge handling: 10 m inward buffer on parcel polygons.")
pipeline_progress.end_stage()


## 4. Prepare the collection round

Resume original saved plans and jobs. Fill the round limit only with previously unselected parcels for deficient crop/partition combinations; locked partitions stay unchanged. Prepare masks only for selected candidates and show the combined workload. Source-specific edge handling is preserved.


In [3]:
pipeline_progress.begin_stage("Prepare collection round")
adaptive_batches = adaptive.prepare_round_batches(
    "francecrops", reference_features, annual_archive_root, report_folder,
    provenance={"dataset_revision": revision, "tables": {name: file_sha256(path) for name, path in table_paths.items()},
                "unavailable_target_classes": ["carrot"], "upstream_splits_preserved": True, "reference_audit": reference_audit},
    inward_buffer_metres=inward_buffer_metres, parcels_per_crop=candidate_parcels_per_crop,
)
annual_plan, adaptive_context = next(((plan, context) for plan, context in adaptive_batches if plan is not None), (None, adaptive_batches[-1][1]))
print("Active/resumed batches:", len(adaptive_batches))
print("Selected parcels:", sum(context["candidate_parcels"] for plan, context in adaptive_batches if plan is not None))
print("Tiles:", sum(plan["estimate"]["tiles"] for plan, context in adaptive_batches if plan is not None))
show_items(sorted({year for plan, context in adaptive_batches if plan is not None for year in plan["years"]}), title="Years")
pipeline_progress.set_workload({"folder": str(annual_archive_root), "source": source_name,
    "plans": [plan["revision"] for plan, context in adaptive_batches if plan is not None],
    "satellite_collection": run_satellite_collection})
pipeline_progress.end_stage()


02_francecrops_preparation: stages:  50%|█████     | 2/4 [00:00<00:00,  3.18stage/s, elapsed 0m 02s; notebook ETA learning (future stages not yet timed)]                   

Adaptive round 1: 150 francecrops candidate parcels; annual histories retained, usable targets checked after four-month selection.
Round 1: target up to 100 candidates per available crop. Resume/submit 1 batches; previous jobs and locked partitions are preserved.
Batch parcels to resume/submit: 150
Estimate for first active batch: {'parcels': 150, 'tiles': 215, 'composites': 5160, 'raw_raster_bytes': 108158400, 'raw_raster_gib': 0.101, 'recommended_free_bytes': 216316800, 'available_bytes': 58437816320, 'note': 'Uncompressed float32 rasters; allow additional disk for summaries, features and models. Compression varies.'}
Pixels by year: {'2022': {'cabbage': 15882, 'potato': 41652}}


## 5. Collect annual imagery

Submit or collect existing Earth Engine jobs. Keep the laptop on until Run All returns normally. Submitted jobs continue in the cloud; rerun later to retrieve results and create annual TIFFs.

Cloud masks, daily mosaics, bilinear band alignment and half-month medians are unchanged. Each completed tile retains 24 periods × seven measurements. One progress display follows the active step; detailed task/page messages are saved to its log. Completion describes saved data, not usable-window quality.


In [5]:
pipeline_progress.begin_stage("Collect annual imagery")
collection_result = adaptive.collect_batches(
    adaptive_batches, earth_engine_project,
    enabled=run_satellite_collection, authenticate=authenticate_now,
    download_workers=download_workers, intervals_per_request=intervals_per_request,
)
pipeline_progress.end_stage("skipped" if not run_satellite_collection else "completed" if collection_result["status"] == "completed" else "pending")


02_francecrops_preparation: Collect annual imagery:  75%|███████▌  | 3/4 [00:04<00:01,  1.34s/stage, elapsed 0m 05s; notebook ETA learning (future stages not yet timed)]

francecrops 2022: selecting tiles 1–215 of 215
Scene inventories: 25/215 (0.1s)
Scene inventories: 50/215 (0.1s)
Scene inventories: 75/215 (0.1s)
Scene inventories: 100/215 (0.2s)
Scene inventories: 125/215 (0.2s)
Scene inventories: 150/215 (0.2s)
Scene inventories: 175/215 (0.3s)
Scene inventories: 200/215 (0.3s)
Scene inventories: 215/215 (0.3s)


02_francecrops_preparation: Collect annual imagery:  75%|███████▌  | 3/4 [00:06<00:02,  2.01s/stage, elapsed 0m 07s; notebook ETA learning (future stages not yet timed)]

  Fetching page 1 (up to 5,000 rows; 0 saved so far)…


02_francecrops_preparation: Collect annual imagery:  75%|███████▌  | 3/4 [00:07<00:02,  2.35s/stage, elapsed 0m 08s; notebook ETA learning (future stages not yet timed)]

  Page 1 verified: 1,000 rows; 1,000 total (0.6s).
  Fetching page 2 (up to 5,000 rows; 1,000 saved so far)…
  Page 2 verified: 1,000 rows; 2,000 total (0.5s).
  Fetching page 3 (up to 5,000 rows; 2,000 saved so far)…


... (305 outputs left out)

02_francecrops_preparation: Collect annual imagery:  75%|███████▌  | 3/4 [02:57<00:59, 59.29s/stage, elapsed 2m 59s; notebook ETA learning (future stages not yet timed)]

annual_14fa2fb77ebe5385253b816f masked daily summaries reused
annual_cbed3e4b3a6f5a7852aa6b02 masked daily summaries reused
annual_841bdf6c5900c83b6fb2fc7b masked daily summaries reused
annual_3929875aad03acb9ce15c06c masked daily summaries reused
annual_4fee49619934bdb25f602be6 masked daily summaries reused
annual_cf5aaa5223bee1310145269b masked daily summaries reused
annual_302b7eccfa478b52d62f48a0 masked daily summaries reused
annual_66825d360b2ae0d250743532 masked daily summaries reused
annual_b632c60b67256d1ffe92753b masked daily summaries reused
annual_94dbc98bfb264b2efe25443d masked daily summaries reused
annual_abb9f0aedcac23557f4bdb27 masked daily summaries reused
annual_46e30a7ac72de57f39e8e4ef masked daily summaries reused
annual_07aff1f3cc3e4f32b011c855 masked daily summaries reused
annual_93cc80b2ff4636a10ab154e9 masked daily summaries reused
annual_35c42be4df02e9c6b2b362e4 masked daily summaries reused
annual_d115c609bdd804d90c45cdf1 masked daily summaries reused
annual_d

02_francecrops_preparation: stages: 100%|██████████| 4/4 [03:04<00:00, 46.10s/stage, elapsed 3m 05s; notebook ETA 0m 00s (future stages not yet timed)]                  

{'status': 'completed'}
02_francecrops_preparation: 3m 05s elapsed; timing report: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/progress/02_francecrops_preparation/runs/20261106T142832_4d53c988.json


## 6. Summary and next action

Check parcel counts, verified tiles, cloud job states and saved paths below. Follow the reported next action. Finish launched batches before [04 Four-month windows](04_four_month_windows.ipynb); it assesses usable samples and deficits. Continue to 05–06 when notebook 04 reports pilot inputs ready, or use strict mode once collection targets are met.

Annual TIFFs are under `annual_tifs/<source>/<year>/<campaign>/round_<number>/`, beside the configured archive root. Masks and Parquet caches remain separate. See [the workflow guide](../docs/workflow.md).


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
source_workflow.print_summary(
    source_name, adaptive_batches, collection_result, reference_mode,
    inward_buffer_metres, reference_note=reference_note,
)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
